# Task 3: A/B Hypothesis Testing

## Objective
Statistically validate or reject key hypotheses about risk drivers, forming the evidence base for ACIS's new segmentation and pricing strategy.

## Risk Metrics
- **Claim Frequency**: Proportion of policies with at least one claim
- **Claim Severity**: Average claim amount, given a claim occurred
- **Margin**: TotalPremium − TotalClaims

## Null Hypotheses to Test
1. H₀: There are no risk differences across provinces
2. H₀: There are no risk differences between zip codes
3. H₀: There is no significant margin (profit) difference between zip codes
4. H₀: There is no significant risk difference between Women and Men

In [1]:
import pandas as pd
import numpy as np
import sys
import os

# Add src directory to path
sys.path.append(os.path.join(os.path.abspath(''), '..'))

from src.hypothesis_tests import (
    calculate_kpis,
    segment_data,
    run_hypothesis_test,
    format_results,
    check_statistical_equivalence,
    generate_business_interpretation
)

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

Libraries imported successfully


## 1. Load and Prepare Data

In [2]:
# Load data
df = pd.read_csv('../data/insurance_data.csv', sep='|', low_memory=False)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

Dataset shape: (1000098, 52)

Columns: ['UnderwrittenCoverID', 'PolicyID', 'TransactionMonth', 'IsVATRegistered', 'Citizenship', 'LegalType', 'Title', 'Language', 'Bank', 'AccountType', 'MaritalStatus', 'Gender', 'Country', 'Province', 'PostalCode', 'MainCrestaZone', 'SubCrestaZone', 'ItemType', 'mmcode', 'VehicleType', 'RegistrationYear', 'make', 'Model', 'Cylinders', 'cubiccapacity', 'kilowatts', 'bodytype', 'NumberOfDoors', 'VehicleIntroDate', 'CustomValueEstimate', 'AlarmImmobiliser', 'TrackingDevice', 'CapitalOutstanding', 'NewVehicle', 'WrittenOff', 'Rebuilt', 'Converted', 'CrossBorder', 'NumberOfVehiclesInFleet', 'SumInsured', 'TermFrequency', 'CalculatedPremiumPerTerm', 'ExcessSelected', 'CoverCategory', 'CoverType', 'CoverGroup', 'Section', 'Product', 'StatutoryClass', 'StatutoryRiskType', 'TotalPremium', 'TotalClaims']


In [3]:
# Calculate KPIs
df = calculate_kpis(df)

print("KPIs calculated:")
print(f"- Claim Frequency: {(df['claim_frequency'] == 1).sum()} claims out of {len(df)} policies ({(df['claim_frequency'] == 1).sum()/len(df)*100:.2f}%)")
print(f"- Claim Severity: Mean = R{df[df['TotalClaims'] > 0]['claim_severity'].mean():.2f}")
print(f"- Margin: Mean = R{df['margin'].mean():.2f}")

KPIs calculated:
- Claim Frequency: 2788 claims out of 1000098 policies (0.28%)
- Claim Severity: Mean = R23273.39
- Margin: Mean = R-2.96


In [4]:
# Explore key features
print("\nProvince distribution:")
print(df['Province'].value_counts())

print("\nGender distribution:")
print(df['Gender'].value_counts())

print("\nPostalCode distribution (top 10):")
print(df['PostalCode'].value_counts().head(10))


Province distribution:
Province
Gauteng          393865
Western Cape     170796
KwaZulu-Natal    169781
North West       143287
Mpumalanga        52718
Eastern Cape      30336
Limpopo           24836
Free State         8099
Northern Cape      6380
Name: count, dtype: int64

Gender distribution:
Gender
Not specified    940990
Male              42817
Female             6755
Name: count, dtype: int64

PostalCode distribution (top 10):
PostalCode
2000    133498
122      49171
7784     28585
299      25546
7405     18518
458      13775
8000     11794
2196     11048
470      10226
7100     10161
Name: count, dtype: int64


## 2. Hypothesis Test 1: Risk Differences Across Provinces

**H₀: There are no risk differences across provinces**

**KPI: Claim Frequency**

**Approach**: Compare Gauteng (largest province) vs Western Cape (second largest)

In [5]:
# Check statistical equivalence on covariates
covariates = ['Gender', 'VehicleType', 'CoverType', 'SumInsured']
equivalence = check_statistical_equivalence(
    df, 'Province', 'Gauteng', 'Western Cape', covariates
)

print("Statistical Equivalence Check (Gauteng vs Western Cape):")
for cov, is_equiv in equivalence.items():
    status = "✓ Equivalent" if is_equiv else "✗ Not equivalent"
    print(f"  {cov}: {status}")

Statistical Equivalence Check (Gauteng vs Western Cape):
  Gender: ✗ Not equivalent
  VehicleType: ✗ Not equivalent
  CoverType: ✗ Not equivalent
  SumInsured: ✓ Equivalent


In [6]:
# Segment data
gauteng_df, western_cape_df = segment_data(df, 'Province', 'Gauteng', 'Western Cape')

print(f"Gauteng: {len(gauteng_df)} policies")
print(f"Western Cape: {len(western_cape_df)} policies")

Gauteng: 393865 policies
Western Cape: 170796 policies


In [7]:
# Run hypothesis test for Claim Frequency
results_province = run_hypothesis_test(
    gauteng_df, western_cape_df, 'claim_frequency', test_type='z_test'
)

print("Hypothesis Test 1: Risk Differences Across Provinces")
print("="*60)
print(format_results(results_province))

Hypothesis Test 1: Risk Differences Across Provinces
Test: Z-test for proportions
P-value: 0.000000
Decision: Reject H₀ (α = 0.05)
Control Group N: 393865
Test Group N: 170796
Z-statistic: -7.5157
Control Proportion: 0.0034
Test Proportion: 0.0022
Difference: -0.0012


In [8]:
# Calculate effect size
gauteng_freq = gauteng_df['claim_frequency'].mean()
western_cape_freq = western_cape_df['claim_frequency'].mean()
effect_size = (gauteng_freq - western_cape_freq) / western_cape_freq

print(f"\nGauteng Claim Frequency: {gauteng_freq:.4f} ({gauteng_freq*100:.2f}%)")
print(f"Western Cape Claim Frequency: {western_cape_freq:.4f} ({western_cape_freq*100:.2f}%)")
print(f"Relative Difference: {effect_size:.2%}")


Gauteng Claim Frequency: 0.0034 (0.34%)
Western Cape Claim Frequency: 0.0022 (0.22%)
Relative Difference: 54.94%


In [9]:
# Generate business interpretation
interpretation_province = generate_business_interpretation(
    hypothesis="No risk differences across provinces",
    results=results_province,
    feature_name="Province",
    control_name="Western Cape",
    test_name="Gauteng",
    kpi_name="Claim Frequency",
    effect_size=effect_size
)

print(f"\nBusiness Interpretation:")
print(interpretation_province)


Business Interpretation:
We reject H₀ for Province (p < 0.05). Gauteng exhibits a 54.9% claim frequency difference compared to Western Cape, suggesting that province is a significant risk driver. A pricing adjustment based on province may be warranted.


## 3. Hypothesis Test 2: Risk Differences Between Zip Codes

**H₀: There are no risk differences between zip codes**

**KPI: Claim Severity**

**Approach**: Compare two large postal codes with similar characteristics

In [10]:
# Find postal codes with sufficient data
postal_counts = df['PostalCode'].value_counts()
large_postals = postal_counts[postal_counts >= 1000].index.tolist()

print(f"Postal codes with >= 1000 policies: {len(large_postals)}")
print(f"Top 10: {large_postals[:10]}")

Postal codes with >= 1000 policies: 197
Top 10: [2000, 122, 7784, 299, 7405, 458, 8000, 2196, 470, 7100]


In [11]:
# Select two postal codes for comparison
postal_a = large_postals[0]
postal_b = large_postals[1]

print(f"Comparing Postal Code {postal_a} vs {postal_b}")

Comparing Postal Code 2000 vs 122


In [12]:
# Check statistical equivalence
equivalence_postal = check_statistical_equivalence(
    df, 'PostalCode', str(postal_a), str(postal_b), covariates
)

print(f"Statistical Equivalence Check (Postal {postal_a} vs {postal_b}):")
for cov, is_equiv in equivalence_postal.items():
    status = "✓ Equivalent" if is_equiv else "✗ Not equivalent"
    print(f"  {cov}: {status}")

Statistical Equivalence Check (Postal 2000 vs 122):
  Gender: ✗ Not equivalent
  VehicleType: ✗ Not equivalent
  CoverType: ✗ Not equivalent
  SumInsured: ✗ Not equivalent


In [13]:
# Segment data
postal_a_df, postal_b_df = segment_data(df, 'PostalCode', str(postal_a), str(postal_b))

# Filter to only policies with claims for severity analysis
postal_a_claims = postal_a_df[postal_a_df['TotalClaims'] > 0]
postal_b_claims = postal_b_df[postal_b_df['TotalClaims'] > 0]

print(f"Postal {postal_a}: {len(postal_a_df)} policies, {len(postal_a_claims)} with claims")
print(f"Postal {postal_b}: {len(postal_b_df)} policies, {len(postal_b_claims)} with claims")

Postal 2000: 0 policies, 0 with claims
Postal 122: 0 policies, 0 with claims


In [14]:
# Run hypothesis test for Claim Severity
results_postal_severity = run_hypothesis_test(
    postal_a_claims, postal_b_claims, 'claim_severity', test_type='t_test'
)

print("Hypothesis Test 2: Risk Differences Between Zip Codes (Claim Severity)")
print("="*60)
print(format_results(results_postal_severity))

Hypothesis Test 2: Risk Differences Between Zip Codes (Claim Severity)
Test: Welch's t-test
P-value: nan
Decision: Fail to reject H₀ (α = 0.05)
Control Group N: 0
Test Group N: 0
T-statistic: nan
Control Mean: nan
Test Mean: nan
Difference: nan


In [15]:
# Calculate effect size
severity_a = postal_a_claims['claim_severity'].mean()
severity_b = postal_b_claims['claim_severity'].mean()
effect_size_severity = (severity_a - severity_b) / severity_b if severity_b > 0 else 0

print(f"\nPostal {postal_a} Claim Severity: R{severity_a:.2f}")
print(f"Postal {postal_b} Claim Severity: R{severity_b:.2f}")
print(f"Relative Difference: {effect_size_severity:.2%}")


Postal 2000 Claim Severity: Rnan
Postal 122 Claim Severity: Rnan
Relative Difference: 0.00%


In [16]:
# Generate business interpretation
interpretation_postal_severity = generate_business_interpretation(
    hypothesis="No risk differences between zip codes",
    results=results_postal_severity,
    feature_name="Postal Code",
    control_name=f"Postal {postal_b}",
    test_name=f"Postal {postal_a}",
    kpi_name="Claim Severity",
    effect_size=effect_size_severity
)

print(f"\nBusiness Interpretation:")
print(interpretation_postal_severity)


Business Interpretation:
We fail to reject H₀ for Postal Code (p = nan). There is no statistically significant difference in claim severity between Postal 2000 and Postal 122. Current pricing for postal code appears adequate.


## 4. Hypothesis Test 3: Margin Differences Between Zip Codes

**H₀: There is no significant margin (profit) difference between zip codes**

**KPI: Margin**

**Approach**: Compare the same two postal codes for margin differences

In [17]:
# Run hypothesis test for Margin
results_postal_margin = run_hypothesis_test(
    postal_a_df, postal_b_df, 'margin', test_type='t_test'
)

print("Hypothesis Test 3: Margin Differences Between Zip Codes")
print("="*60)
print(format_results(results_postal_margin))

Hypothesis Test 3: Margin Differences Between Zip Codes
Test: Welch's t-test
P-value: nan
Decision: Fail to reject H₀ (α = 0.05)
Control Group N: 0
Test Group N: 0
T-statistic: nan
Control Mean: nan
Test Mean: nan
Difference: nan


In [18]:
# Calculate effect size
margin_a = postal_a_df['margin'].mean()
margin_b = postal_b_df['margin'].mean()
effect_size_margin = (margin_a - margin_b) / abs(margin_b) if margin_b != 0 else 0

print(f"\nPostal {postal_a} Margin: R{margin_a:.2f}")
print(f"Postal {postal_b} Margin: R{margin_b:.2f}")
print(f"Relative Difference: {effect_size_margin:.2%}")


Postal 2000 Margin: Rnan
Postal 122 Margin: Rnan
Relative Difference: nan%


In [19]:
# Generate business interpretation
interpretation_postal_margin = generate_business_interpretation(
    hypothesis="No significant margin difference between zip codes",
    results=results_postal_margin,
    feature_name="Postal Code",
    control_name=f"Postal {postal_b}",
    test_name=f"Postal {postal_a}",
    kpi_name="Margin",
    effect_size=effect_size_margin
)

print(f"\nBusiness Interpretation:")
print(interpretation_postal_margin)


Business Interpretation:
We fail to reject H₀ for Postal Code (p = nan). There is no statistically significant difference in margin between Postal 2000 and Postal 122. Current pricing for postal code appears adequate.


## 5. Hypothesis Test 4: Risk Differences Between Women and Men

**H₀: There is no significant risk difference between Women and Men**

**KPI: Claim Frequency**

**Approach**: Compare Male vs Female (excluding 'Not specified')

In [20]:
# Filter to only Male and Female
gender_df = df[df['Gender'].isin(['Male', 'Female'])].copy()

print(f"Policies with Gender specified: {len(gender_df)}")
print(gender_df['Gender'].value_counts())

Policies with Gender specified: 49572
Gender
Male      42817
Female     6755
Name: count, dtype: int64


In [21]:
# Check statistical equivalence
equivalence_gender = check_statistical_equivalence(
    gender_df, 'Gender', 'Male', 'Female', ['Province', 'VehicleType', 'CoverType', 'SumInsured']
)

print("Statistical Equivalence Check (Male vs Female):")
for cov, is_equiv in equivalence_gender.items():
    status = "✓ Equivalent" if is_equiv else "✗ Not equivalent"
    print(f"  {cov}: {status}")

Statistical Equivalence Check (Male vs Female):
  Province: ✗ Not equivalent
  VehicleType: ✗ Not equivalent
  CoverType: ✗ Not equivalent
  SumInsured: ✓ Equivalent


In [22]:
# Segment data
male_df, female_df = segment_data(gender_df, 'Gender', 'Male', 'Female')

print(f"Male: {len(male_df)} policies")
print(f"Female: {len(female_df)} policies")

Male: 42817 policies
Female: 6755 policies


In [23]:
# Run hypothesis test for Claim Frequency
results_gender = run_hypothesis_test(
    male_df, female_df, 'claim_frequency', test_type='z_test'
)

print("Hypothesis Test 4: Risk Differences Between Women and Men")
print("="*60)
print(format_results(results_gender))

Hypothesis Test 4: Risk Differences Between Women and Men
Test: Z-test for proportions
P-value: 0.840494
Decision: Fail to reject H₀ (α = 0.05)
Control Group N: 42817
Test Group N: 6755
Z-statistic: -0.2013
Control Proportion: 0.0022
Test Proportion: 0.0021
Difference: -0.0001


In [24]:
# Calculate effect size
male_freq = male_df['claim_frequency'].mean()
female_freq = female_df['claim_frequency'].mean()
effect_size_gender = (male_freq - female_freq) / female_freq if female_freq > 0 else 0

print(f"\nMale Claim Frequency: {male_freq:.4f} ({male_freq*100:.2f}%)")
print(f"Female Claim Frequency: {female_freq:.4f} ({female_freq*100:.2f}%)")
print(f"Relative Difference: {effect_size_gender:.2%}")


Male Claim Frequency: 0.0022 (0.22%)
Female Claim Frequency: 0.0021 (0.21%)
Relative Difference: 5.93%


In [25]:
# Generate business interpretation
interpretation_gender = generate_business_interpretation(
    hypothesis="No significant risk difference between Women and Men",
    results=results_gender,
    feature_name="Gender",
    control_name="Female",
    test_name="Male",
    kpi_name="Claim Frequency",
    effect_size=effect_size_gender
)

print(f"\nBusiness Interpretation:")
print(interpretation_gender)


Business Interpretation:
We fail to reject H₀ for Gender (p = 0.8405). There is no statistically significant difference in claim frequency between Male and Female. Current pricing for gender appears adequate.


## 6. Results Summary Table

In [26]:
# Create results summary table
results_summary = pd.DataFrame([
    {
        'Hypothesis': 'H₀: No risk differences across provinces',
        'KPI': 'Claim Frequency',
        'Groups Compared': 'Gauteng vs Western Cape',
        'Test Used': results_province['test_name'],
        'P-value': f"{results_province['p_value']:.6f}",
        'Decision': results_province['decision'],
        'Effect Size': f"{effect_size:.2%}"
    },
    {
        'Hypothesis': 'H₀: No risk differences between zip codes',
        'KPI': 'Claim Severity',
        'Groups Compared': f'Postal {postal_a} vs {postal_b}',
        'Test Used': results_postal_severity['test_name'],
        'P-value': f"{results_postal_severity['p_value']:.6f}",
        'Decision': results_postal_severity['decision'],
        'Effect Size': f"{effect_size_severity:.2%}"
    },
    {
        'Hypothesis': 'H₀: No significant margin difference between zip codes',
        'KPI': 'Margin',
        'Groups Compared': f'Postal {postal_a} vs {postal_b}',
        'Test Used': results_postal_margin['test_name'],
        'P-value': f"{results_postal_margin['p_value']:.6f}",
        'Decision': results_postal_margin['decision'],
        'Effect Size': f"{effect_size_margin:.2%}"
    },
    {
        'Hypothesis': 'H₀: No risk difference between Women and Men',
        'KPI': 'Claim Frequency',
        'Groups Compared': 'Male vs Female',
        'Test Used': results_gender['test_name'],
        'P-value': f"{results_gender['p_value']:.6f}",
        'Decision': results_gender['decision'],
        'Effect Size': f"{effect_size_gender:.2%}"
    }
])

print("Hypothesis Testing Results Summary")
print("="*100)
print(results_summary.to_string(index=False))

Hypothesis Testing Results Summary
                                            Hypothesis             KPI         Groups Compared              Test Used  P-value          Decision Effect Size
              H₀: No risk differences across provinces Claim Frequency Gauteng vs Western Cape Z-test for proportions 0.000000         Reject H₀      54.94%
             H₀: No risk differences between zip codes  Claim Severity      Postal 2000 vs 122         Welch's t-test      nan Fail to reject H₀       0.00%
H₀: No significant margin difference between zip codes          Margin      Postal 2000 vs 122         Welch's t-test      nan Fail to reject H₀        nan%
          H₀: No risk difference between Women and Men Claim Frequency          Male vs Female Z-test for proportions 0.840494 Fail to reject H₀       5.93%


## 7. Business Recommendations

In [27]:
print("BUSINESS RECOMMENDATIONS")
print("="*100)
print()

print("1. Province-based Risk Adjustment:")
print("   " + interpretation_province)
print()

print("2. Postal Code-based Risk Adjustment (Claim Severity):")
print("   " + interpretation_postal_severity)
print()

print("3. Postal Code-based Margin Adjustment:")
print("   " + interpretation_postal_margin)
print()

print("4. Gender-based Risk Adjustment:")
print("   " + interpretation_gender)

BUSINESS RECOMMENDATIONS

1. Province-based Risk Adjustment:
   We reject H₀ for Province (p < 0.05). Gauteng exhibits a 54.9% claim frequency difference compared to Western Cape, suggesting that province is a significant risk driver. A pricing adjustment based on province may be warranted.

2. Postal Code-based Risk Adjustment (Claim Severity):
   We fail to reject H₀ for Postal Code (p = nan). There is no statistically significant difference in claim severity between Postal 2000 and Postal 122. Current pricing for postal code appears adequate.

3. Postal Code-based Margin Adjustment:
   We fail to reject H₀ for Postal Code (p = nan). There is no statistically significant difference in margin between Postal 2000 and Postal 122. Current pricing for postal code appears adequate.

4. Gender-based Risk Adjustment:
   We fail to reject H₀ for Gender (p = 0.8405). There is no statistically significant difference in claim frequency between Male and Female. Current pricing for gender appears 

## 8. Save Results

In [28]:
# Save results summary to CSV
results_summary.to_csv('../hypothesis_test_results.csv', index=False)
print("Results saved to hypothesis_test_results.csv")

Results saved to hypothesis_test_results.csv


## Conclusion

This analysis provides statistical evidence for or against risk differences across key segmentation variables. The results inform ACIS's new segmentation and pricing strategy by identifying which factors significantly impact risk metrics.